# Section 7 — CNN regime 1 (Autoencoder) · Compare Results
Loads the 5 methods' `results/*.json` from Drive (missing ones skipped). BCE + pixel-accuracy, cost/memory, three-factor cos-sweep head-to-head, and a sample-reconstruction grid.

## 1. Setup + Load Results

In [ ]:
import os, json, math, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
device='cuda' if torch.cuda.is_available() else 'cpu'
USE_DRIVE, DRIVE_SUBDIR = True, 'Section7_r1_autoencoder'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); STORE=os.path.join('/content/drive/MyDrive',DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed:',e); STORE=os.path.join('/content',DRIVE_SUBDIR)
else:
    STORE=os.path.join('.',DRIVE_SUBDIR)
RESULTS_DIR=os.path.join(STORE,'results'); CKPT_DIR=os.path.join(STORE,'checkpoints')
METHODS=['backprop','two_factor','three_factor_clean','three_factor_normal','three_factor_noisy', 'backprop_100M']
LABELS={'backprop':'Backprop','two_factor':'Two-factor Hebbian','three_factor_clean':'Three-factor clean (cos~0.5)','three_factor_normal':'Three-factor normal (cos~0.09)','three_factor_noisy':'Three-factor noisy (cos~0.01)', 'backprop_100M':'Backprop 100M (ceiling)'}
COLORS={'backprop':'#1F3864','two_factor':'#B8860B','three_factor_clean':'#C62828','three_factor_normal':'#E67E22','three_factor_noisy':'#7B1FA2', 'backprop_100M':'#2E7D32'}
R={}
for m in METHODS:
    p=os.path.join(RESULTS_DIR,f'{m}.json')
    if os.path.exists(p):
        R[m]=json.load(open(p)); s=R[m]['summary']; md=R[m]['meta']
        print(f'loaded {m:24} {md["total_steps"]:>9,} steps  {md["wall_clock_sec"]/3600:5.2f}h  best BCE {s["best_bce"]:.4f}  best pixel-acc {s["best_acc"]:.4f}')
    else:
        print(f'MISSING {p} (run the {m} notebook first)')

## 2. Compare curves (BCE + pixel-acc)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m in R:
    c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
    ax[0].plot(hrs, c['test_bce'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[1].plot(hrs, c['test_acc'], 'o-', color=COLORS[m], label=LABELS[m])
ax[0].set_xlabel('hours'); ax[0].set_ylabel('test BCE / pixel'); ax[0].set_title('Reconstruction BCE vs time'); ax[0].legend()
if R:
    bg = list(R.values())[0]['summary'].get('bg_baseline');
    if bg: ax[1].axhline(bg, color='gray', ls='--', lw=1, label=f'all-bg {bg:.2f}')
ax[1].set_xlabel('hours'); ax[1].set_ylabel('pixel accuracy'); ax[1].set_title('Pixel-accuracy vs time'); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Cost & Memory

In [ ]:
def fwd_equiv(m):
    st = R[m]['meta']['total_steps']
    if m.startswith('three_factor'): return st * 2 * R[m]['meta']['config'].get('M', 0)
    if m.startswith('backprop'): return st * 3
    return st * 2
print(f'{"method":30}{"steps":>10}{"fwd-equiv":>14}{"wall h":>8}{"peak MB":>9}')
print('-'*71)
for m in R:
    md = R[m]['meta']
    print(f'{LABELS[m]:30}{md["total_steps"]:>10,}{fwd_equiv(m):>14,}{md["wall_clock_sec"]/3600:>8.2f}{md.get("peak_mem_mb", float("nan")):>9.1f}')

## 4. Head-to-head — three-factor cos sweep

In [ ]:
tf = [m for m in ['three_factor_clean','three_factor_normal','three_factor_noisy'] if m in R]
if len(tf) >= 2:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for m in tf:
        c = R[m]['curve']; hrs=[t/3600 for t in c['t_sec']]
        ax[0].plot(hrs, c['test_bce'], 'o-', color=COLORS[m], label=LABELS[m])
        ax[1].plot(hrs, c['test_acc'], 'o-', color=COLORS[m], label=LABELS[m])
        ax[2].plot(c['step'], c['test_bce'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[0].set_xlabel('hours'); ax[0].set_ylabel('test BCE'); ax[0].set_title('cos sweep — BCE vs time'); ax[0].legend()
    ax[1].set_xlabel('hours'); ax[1].set_ylabel('pixel acc'); ax[1].set_title('pixel-acc vs time'); ax[1].legend()
    ax[2].set_xscale('symlog'); ax[2].set_xlabel('steps'); ax[2].set_ylabel('test BCE'); ax[2].set_title('BCE vs steps'); ax[2].legend()
    plt.tight_layout(); plt.show()
    print(f'{"variant":30}{"M":>10}{"cos~":>8}{"steps":>10}{"best_bce":>11}{"best_acc":>10}')
    print('-'*79)
    for m in tf:
        md, s = R[m]['meta'], R[m]['summary']; M = md['config'].get('M') or 0; P = md['P']
        cos = (M/(M+P+1))**0.5 if M else float('nan')
        print(f'{LABELS[m]:30}{M:>10,}{cos:>8.3f}{md["total_steps"]:>10,}{s["best_bce"]:>11.4f}{s["best_acc"]:>10.4f}')
    best = min(tf, key=lambda m: R[m]['summary']['best_bce'])
    print(f'\nLowest best BCE in the budget: {LABELS[best]}.')
else:
    print('Head-to-head needs >=2 three-factor variants (03/04/05).')

## 5. Sample reconstructions (best model)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvAE(nn.Module):
    """Conv encoder -> dense latent bottleneck -> Upsample+Conv decoder -> 28x28 logits.
    No BatchNorm (vmap-clean); Upsample+Conv instead of ConvTranspose (vmap-safe)."""
    def __init__(self, conv_c=48, latent=98):
        super().__init__()
        c2 = 2 * conv_c
        self.enc = nn.Sequential(
            nn.Conv2d(1, conv_c, 3, stride=2, padding=1), nn.ReLU(),   # 28 -> 14
            nn.Conv2d(conv_c, c2, 3, stride=2, padding=1), nn.ReLU(),  # 14 -> 7
        )
        self._c2 = c2
        self.fc1 = nn.Linear(c2 * 7 * 7, latent)
        self.fc2 = nn.Linear(latent, c2 * 7 * 7)
        self.dec = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),               # 7 -> 14
            nn.Conv2d(c2, conv_c, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),               # 14 -> 28
            nn.Conv2d(conv_c, 1, 3, padding=1),
        )
    def forward(self, x):
        h = self.enc(x)
        b = h.shape[0]
        z = self.fc1(h.reshape(b, -1))
        h2 = self.fc2(z).reshape(b, self._c2, 7, 7)
        return self.dec(h2)                                            # (B,1,28,28) logits

# sample reconstructions for the best model: input | target | reconstruction
import torch
if R:
    best = max(R, key=lambda m: R[m]['summary']['best_acc'])
    cfg = R[best]['meta']['config']
    net = ConvAE(cfg['CONV_C'], cfg['LATENT']).to(device)
    ck = torch.load(os.path.join(CKPT_DIR, f'{best}.pt'), map_location=device)['method']
    net.load_state_dict(ck['net'] if 'net' in ck else ck['params']); net.eval()
    import torchvision
    _te = torchvision.datasets.MNIST('./data', train=False, download=True)
    x = (_te.data[:8].float()/255.0 > 0.5).float().unsqueeze(1).to(device)
    torch.manual_seed(cfg['SEED']); xin = x + cfg['CORRUPT_SIGMA']*torch.randn_like(x) if cfg['CORRUPT_SIGMA']>0 else x
    with torch.no_grad(): rec = (torch.sigmoid(net(xin)) > 0.5).float()
    fig, ax = plt.subplots(3, 8, figsize=(12, 4.6))
    for j in range(8):
        ax[0,j].imshow(xin[j,0].cpu(), cmap='gray'); ax[1,j].imshow(x[j,0].cpu(), cmap='gray'); ax[2,j].imshow(rec[j,0].cpu(), cmap='gray')
        for r in range(3): ax[r,j].axis('off')
    ax[0,0].set_ylabel('input'); ax[1,0].set_ylabel('target'); ax[2,0].set_ylabel('reconstruction')
    plt.suptitle(f'Best model: {LABELS[best]} (pixel-acc {R[best]["summary"]["best_acc"]:.3f})'); plt.tight_layout(); plt.show()
else:
    print('no results loaded')

## 6. Summary table

In [ ]:
print(f'{"Experiment":30}{"Steps":>10}{"Init BCE":>10}{"Best BCE":>10}{"Final acc":>11}{"Best acc":>10}')
print('-'*81)
for m in R:
    md, s = R[m]['meta'], R[m]['summary']
    print(f'{LABELS[m]:30}{md["total_steps"]:>10,}{s["initial_bce"]:>10.4f}{s["best_bce"]:>10.4f}{s["final_acc"]:>11.4f}{s["best_acc"]:>10.4f}')
print(f'\n(all-background pixel-acc baseline = {list(R.values())[0]["summary"]["bg_baseline"]:.3f} — methods must beat this)' if R else '')

## 7. Full autoencoder results — per-method reconstructions + metrics

In [ ]:
# ── Full autoencoder results: per-method reconstructions + foreground-aware metrics ──
# Pixel-accuracy sits on the ~0.87 all-background floor, so it barely separates methods.
# We add the metrics that actually measure reconstruction of the "ink":
#   foreground IoU, Dice / F1, precision, recall, balanced-acc — plus test BCE.
# Each method's net is rebuilt FROM ITS OWN saved config, so this also handles the 100M ceiling.
import os, csv, torch, torchvision
import torch.nn.functional as F
import numpy as np

def _ckpt_path(m):
    for d in [CKPT_DIR] + list(globals().get('EXTRA_CKPT_DIRS', [])):
        pth = os.path.join(d, f'{m}.pt')
        if os.path.exists(pth): return pth
    return None

N_EVAL, N_SHOW = 2000, 8
_te  = torchvision.datasets.MNIST('./data', train=False, download=True)
Xtgt = (_te.data.float()/255.0 > 0.5).float().unsqueeze(1)[:N_EVAL].to(device)   # (N,1,28,28)

RECON_METRICS, recons = [], {}
for m in METHODS:
    if m not in R: continue
    cp = _ckpt_path(m)
    if cp is None:
        print(f'skip {m:24} (no checkpoint on this Drive — run it or merge from the other account)'); continue
    cfg = R[m]['meta']['config']
    net = ConvAE(cfg['CONV_C'], cfg['LATENT']).to(device)
    ck  = torch.load(cp, map_location=device)['method']
    net.load_state_dict(ck['net'] if 'net' in ck else ck['params']); net.eval()
    sig = cfg.get('CORRUPT_SIGMA', 0.0)
    torch.manual_seed(cfg.get('SEED', 0))
    Xin = Xtgt + sig*torch.randn_like(Xtgt) if sig > 0 else Xtgt
    with torch.no_grad():
        logit = net(Xin)
        bce   = F.binary_cross_entropy_with_logits(logit, Xtgt).item()
        pred  = (torch.sigmoid(logit) > 0.5).float()
    t, p = Xtgt, pred
    tp = (p*t).sum().item(); fp = (p*(1-t)).sum().item(); fn = ((1-p)*t).sum().item(); tn = ((1-p)*(1-t)).sum().item()
    pix_acc = (tp+tn)/(tp+tn+fp+fn+1e-9)
    fg_iou  = tp/(tp+fp+fn+1e-9)
    dice    = 2*tp/(2*tp+fp+fn+1e-9)                       # = F1 on foreground pixels
    prec, rec = tp/(tp+fp+1e-9), tp/(tp+fn+1e-9)
    bal_acc = 0.5*(tp/(tp+fn+1e-9) + tn/(tn+fp+1e-9))
    RECON_METRICS.append([m, R[m]['meta']['P'], round(pix_acc,4), round(fg_iou,4), round(dice,4),
                          round(prec,3), round(rec,3), round(bal_acc,4), round(bce,4)])
    recons[m] = pred[:N_SHOW].cpu()

# --- metrics table ---
if RECON_METRICS:
    print(f'{"method":30}{"P":>12}{"pix-acc":>9}{"fg-IoU":>8}{"Dice/F1":>9}{"prec":>7}{"recall":>8}{"bal-acc":>9}{"BCE":>8}')
    print('-'*108)
    for r in RECON_METRICS:
        print(f'{LABELS[r[0]]:30}{r[1]:>12,}{r[2]:>9.4f}{r[3]:>8.4f}{r[4]:>9.4f}{r[5]:>7.3f}{r[6]:>8.3f}{r[7]:>9.4f}{r[8]:>8.4f}')
    _bg = list(R.values())[0]['summary'].get('bg_baseline')
    if _bg: print(f'\nall-background pixel-acc floor = {_bg:.3f}  →  fg-IoU / Dice are floor-free; they show the REAL reconstruction quality.')
else:
    print('No checkpoints available yet — run the method notebooks (or merge checkpoints) first.')

# persist to the export folder so the zip picks them up
_EXPORT = os.path.join(STORE, 'exports'); os.makedirs(_EXPORT, exist_ok=True)
if RECON_METRICS:
    with open(os.path.join(_EXPORT, 'reconstruction_metrics.csv'), 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['method','P','pixel_acc','fg_iou','dice_f1','precision','recall','balanced_acc','bce'])
        for r in RECON_METRICS: w.writerow(r)

# --- reconstruction grid: top row = target, one row per method ---
if recons:
    order = [m for m in METHODS if m in recons]
    nrow  = 1 + len(order)
    fig, ax = plt.subplots(nrow, N_SHOW, figsize=(1.4*N_SHOW, 1.35*nrow))
    if nrow == 1: ax = ax[None, :]
    for j in range(N_SHOW):
        ax[0, j].imshow(Xtgt[j,0].cpu(), cmap='gray'); ax[0, j].axis('off')
    ax[0, 0].text(-0.30, 0.5, 'target', fontsize=9, rotation=90, va='center', ha='center',
                  transform=ax[0,0].transAxes, fontweight='bold')
    for r, m in enumerate(order, start=1):
        for j in range(N_SHOW):
            ax[r, j].imshow(recons[m][j,0], cmap='gray'); ax[r, j].axis('off')
        ax[r, 0].text(-0.30, 0.5, LABELS[m].split(' (')[0], fontsize=8, rotation=90, va='center',
                      ha='center', transform=ax[r,0].transAxes, color=COLORS[m])
    plt.suptitle('Per-method reconstructions (top row = target)'); plt.tight_layout()
    fig.savefig(os.path.join(_EXPORT, 'reconstructions_grid.png'), dpi=130, bbox_inches='tight')
    plt.show()

# --- bar: pixel-acc (background-inflated) vs foreground IoU (real quality) ---
if RECON_METRICS:
    ms = [r[0] for r in RECON_METRICS]; xs = np.arange(len(ms)); w = 0.38
    fig, axb = plt.subplots(figsize=(1.7*len(ms)+3, 4))
    axb.bar(xs-w/2, [r[2] for r in RECON_METRICS], w, label='pixel-acc', color='#9AA5B1')
    axb.bar(xs+w/2, [r[3] for r in RECON_METRICS], w, label='fg-IoU', color=[COLORS[m] for m in ms])
    _bg = list(R.values())[0]['summary'].get('bg_baseline')
    if _bg: axb.axhline(_bg, color='gray', ls='--', lw=1, label=f'all-bg {_bg:.2f}')
    axb.set_xticks(xs); axb.set_xticklabels([LABELS[m].split(' (')[0] for m in ms], rotation=30, ha='right', fontsize=8)
    axb.set_ylabel('score'); axb.set_ylim(0, 1)
    axb.set_title('Pixel-acc (background-inflated) vs foreground IoU'); axb.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(_EXPORT, 'metrics_pixacc_vs_fgiou.png'), dpi=130, bbox_inches='tight')
    plt.show()

## 8. Export figures + CSV → Drive

In [ ]:
# ── Export §7-AE comparison: every figure (PNG) + data (CSV) → a Drive folder, then download a zip ──
import os, csv, glob, shutil
EXPORT_DIR = os.path.join(STORE, 'exports'); os.makedirs(EXPORT_DIR, exist_ok=True)

# 1) CSVs — long-format curves + one summary row per method (robust to differing keys across methods)
if R:
    ckeys = []
    for m in R:
        for k in R[m]['curve']:
            if k not in ckeys: ckeys.append(k)
    with open(os.path.join(EXPORT_DIR, 'curves.csv'), 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['method'] + ckeys)
        for m in R:
            c = R[m]['curve']; n = len(c.get('t_sec', []))
            for i in range(n):
                w.writerow([m] + [c[k][i] if (k in c and i < len(c[k])) else '' for k in ckeys])
    skeys = []
    for m in R:
        for k in R[m]['summary']:
            if k not in skeys: skeys.append(k)
    with open(os.path.join(EXPORT_DIR, 'summary.csv'), 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['method', 'P', 'total_steps', 'wall_clock_h', 'M', 'cos'] + skeys)
        for m in R:
            md, s = R[m]['meta'], R[m]['summary']; M = md['config'].get('M') or 0
            cos = round((M/(M+md['P']+1))**0.5, 4) if M else ''
            w.writerow([m, md['P'], md['total_steps'], round(md['wall_clock_sec']/3600, 3), M, cos] + [s.get(k, '') for k in skeys])
    print('wrote curves.csv, summary.csv')
if 'RECON_METRICS' in globals() and RECON_METRICS:
    with open(os.path.join(EXPORT_DIR, 'reconstruction_metrics.csv'), 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['method','P','pixel_acc','fg_iou','dice_f1','precision','recall','balanced_acc','bce'])
        for r in RECON_METRICS: w.writerow(r)
    print('wrote reconstruction_metrics.csv')

# 2) Figures — redraw & save the standard comparison plots (reconstruction grid + bar are saved by the cell above)
def _save(fig, name): fig.savefig(os.path.join(EXPORT_DIR, name), dpi=130, bbox_inches='tight'); plt.close(fig)
if R:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for m in R:
        c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
        ax[0].plot(hrs, c['test_bce'], 'o-', color=COLORS[m], label=LABELS[m])
        ax[1].plot(hrs, c['test_acc'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[0].set_xlabel('hours'); ax[0].set_ylabel('test BCE / pixel'); ax[0].set_title('Reconstruction BCE vs time'); ax[0].legend()
    _bg = list(R.values())[0]['summary'].get('bg_baseline')
    if _bg: ax[1].axhline(_bg, color='gray', ls='--', lw=1, label=f'all-bg {_bg:.2f}')
    ax[1].set_xlabel('hours'); ax[1].set_ylabel('pixel accuracy'); ax[1].set_title('Pixel-accuracy vs time'); ax[1].legend()
    _save(fig, 'curves_bce_pixacc.png')
    tf = [m for m in ['three_factor_clean','three_factor_normal','three_factor_noisy'] if m in R]
    if len(tf) >= 2:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
        for m in tf:
            c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
            ax[0].plot(hrs, c['test_bce'], 'o-', color=COLORS[m], label=LABELS[m])
            ax[1].plot(hrs, c['test_acc'], 'o-', color=COLORS[m], label=LABELS[m])
        ax[0].set_xlabel('hours'); ax[0].set_ylabel('test BCE'); ax[0].set_title('cos sweep — BCE'); ax[0].legend()
        ax[1].set_xlabel('hours'); ax[1].set_ylabel('pixel acc'); ax[1].set_title('cos sweep — pixel-acc'); ax[1].legend()
        _save(fig, 'cos_sweep.png')
    print('wrote comparison figures')

# 3) Copy any per-run progress PNGs the trainers saved to Drive
for d in {RESULTS_DIR, CKPT_DIR, STORE}:
    for png in glob.glob(os.path.join(d, '*.png')):
        try: shutil.copy(png, EXPORT_DIR)
        except Exception: pass

# 4) Zip the folder (kept on Drive too) and auto-download it in Colab
print('\nexport folder:', EXPORT_DIR); print(' ', sorted(os.listdir(EXPORT_DIR)))
_ztarget = '/content' if os.path.isdir('/content') else os.path.dirname(EXPORT_DIR)
zip_path = shutil.make_archive(os.path.join(_ztarget, 'Section7_AE_regime1_exports'), 'zip', EXPORT_DIR)
try:
    if os.path.abspath(os.path.dirname(zip_path)) != os.path.abspath(STORE): shutil.copy(zip_path, STORE)
except Exception as e: print('(could not copy zip to Drive):', e)
print('zip:', zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('(download only runs in Colab):', e)